In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
import seaborn as sns # For heatmaps
from skopt import BayesSearchCV
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, train_test_split, cross_validate
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error

In [3]:
df = pd.read_csv('ChEMBL_amines_12C_morgan_fingerprints.csv')
df

,ChEMBL ID,Name,Molecular Weight,CX Acidic pKa,CX Basic pKa,Aromatic Rings,Molecular Species,Molecular Formula,Smiles,Inchi Key,Amine Class,Morgan_Fingerprint
0,CHEMBL4297305,NB-001,264.33,NaN,10.25,2.0,BASE,C12H20N6O,Nc1ncnc2c1ncn2CCNCCCCCO,CVPTTZZCRDVGSU-UHFFFAOYSA-N,secondary,0000000000000000000001000000000000000000000000...
1,CHEMBL1574,PHENTERMINE,149.24,NaN,10.25,1.0,BASE,C10H15N,CC(C)(N)Cc1ccccc1,DHHVAGZRUROJKS-UHFFFAOYSA-N,primary,0000000000000000000000000000000000000000000000...
2,CHEMBL5394915,NaN,258.32,12.58,8.85,0.0,BASE,C12H22N2O4,COC(=O)[C@@H]1CCCN1C(=O)[C@@H](O)[C@H](N)C(C)C,UCLCHJWFVUTOCB-AEJSXWLSSA-N,primary,0100000000000000000000000000000000000000001000...
3,CHEMBL1546,HYDROXYAMPHETAMINE,151.21,10.48,9.80,1.0,BASE,C9H13NO,CC(N)Cc1ccc(O)cc1,GIKNHHRFLCDOEU-UHFFFAOYSA-N,primary,0100000000000000000000000000000000000000000000...
4,CHEMBL2105671,AFEGOSTAT TARTRATE,297.26,13.52,8.80,0.0,BASE,C10H19NO9,O=C(O)C(O)C(O)C(=O)O.OC[C@H]1CNC[C@@H](O)[C@@H]1O,ULBPPCHRAVUQMC-RWOHWRPJSA-N,secondary,0100000000000000000000000000000000000000000000...
...,...,...,...,...,...,...,...,...,...,...,...,...
5862,CHEMBL684,DIETHYLCARBAMAZINE,199.30,NaN,6.90,0.0,NEUTRAL,C10H21N3O,CCN(CC)C(=O)N1CCN(C)CC1,RCKMWOKWVGPNJF-UHFFFAOYSA-N,tertiary,0000000000000010000000000000000000000000000000...
5863,CHEMBL1086997,LUCERASTAT,219.28,12.90,8.49,0.0,NEUTRAL,C10H21NO4,CCCCN1C[C@H](O)[C@@H](O)[C@@H](O)[C@H]1CO,UQRORFVVSGFNRO-XFWSIPNHSA-N,tertiary,0000000000000000000000000000000000000000000000...
5864,CHEMBL511099,BICIFADINE,173.26,NaN,10.60,1.0,BASE,C12H15N,Cc1ccc(C23CNCC2C3)cc1,OFYVIGTWSQPCLF-UHFFFAOYSA-N,secondary,0000000000000000000000000000000000000000000000...
5865,CHEMBL358040,NORFENEFRINE,153.18,9.56,8.91,1.0,BASE,C8H11NO2,NCC(O)c1cccc(O)c1,LRCXRAABFLIVAI-UHFFFAOYSA-N,primary,0100000000000000000000000000000000000000000000...


In [4]:
print("--- Basic Data Overview ---")
# 1. Count the number of rows
num_rows = df.shape[0]
print(f"Number of rows in the dataset: {num_rows}")

--- Basic Data Overview ---
Number of rows in the dataset: 5867


In [5]:
# 2. Count the number of unique Inchi Keys
num_unique_inchi_keys = df['Inchi Key'].nunique()
print(f"Number of unique Inchi Keys: {num_unique_inchi_keys}")

Number of unique Inchi Keys: 5867


In [6]:
# 3. Range of CX Basic pKa
df['CX Basic pKa'] = pd.to_numeric(df['CX Basic pKa'], errors='coerce')
pka_min = df['CX Basic pKa'].min()
pka_max = df['CX Basic pKa'].max()
print(f"Range of CX Basic pKa: {pka_min:.2f} - {pka_max:.2f}")

Range of CX Basic pKa: 0.65 - 12.89


In [7]:
# 4. Distribution of Amine class (primary, secondary, tertiary)
amine_class_distribution = df['Amine Class'].value_counts(normalize=True) * 100
print("\nPercentage distribution of Amine Class:")
print(amine_class_distribution.round(2))


Percentage distribution of Amine Class:
Amine Class
primary      35.90
secondary    34.04
tertiary     30.07
Name: proportion, dtype: float64


In [8]:
# function to canonicalize a SMILES string
def canonicalize_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        return Chem.MolToSmiles(mol, canonical=True)
    return None  # or smi if you want to keep the original on failure

# Apply to the 'smiles' column of your DataFrame
df['canonical_smiles'] = df['Smiles'].apply(canonicalize_smiles)

In [9]:
df.describe()

,Molecular Weight,CX Acidic pKa,CX Basic pKa,Aromatic Rings
count,5867.000000,1893.000000,5867.000000,5867.000000
mean,191.967505,11.738959,8.887055,0.692688
std,39.590318,1.709733,1.382169,0.667408
min,31.060000,2.820000,0.650000,0.000000
25%,168.200000,10.190000,8.220000,0.000000
50%,191.270000,12.400000,9.040000,1.000000
75%,215.210000,13.130000,9.750000,1.000000
max,347.380000,14.000000,12.890000,3.000000


In [10]:
df1 = df[['ChEMBL ID', 'CX Basic pKa', 'canonical_smiles']]
df1

,ChEMBL ID,CX Basic pKa,canonical_smiles
0,CHEMBL4297305,10.25,Nc1ncnc2c1ncn2CCNCCCCCO
1,CHEMBL1574,10.25,CC(C)(N)Cc1ccccc1
2,CHEMBL5394915,8.85,COC(=O)[C@@H]1CCCN1C(=O)[C@@H](O)[C@H](N)C(C)C
3,CHEMBL1546,9.80,CC(N)Cc1ccc(O)cc1
4,CHEMBL2105671,8.80,O=C(O)C(O)C(O)C(=O)O.OC[C@H]1CNC[C@@H](O)[C@@H]1O
...,...,...,...
5862,CHEMBL684,6.90,CCN(CC)C(=O)N1CCN(C)CC1
5863,CHEMBL1086997,8.49,CCCCN1C[C@H](O)[C@@H](O)[C@@H](O)[C@H]1CO
5864,CHEMBL511099,10.60,Cc1ccc(C23CNCC2C3)cc1
5865,CHEMBL358040,8.91,NCC(O)c1cccc(O)c1


In [11]:
# ---------------------------
# Compute Morgan Fingerprints
# ---------------------------
def smiles_to_morgan_fp(smi, radius=2, nBits=1024):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=nBits)
        return list(fp)
    return [0] * nBits

df1['morgan_fp'] = df1['canonical_smiles'].apply(smiles_to_morgan_fp)

/tmp/ipykernel_144/1448783578.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df1['morgan_fp'] = df1['canonical_smiles'].apply(smiles_to_morgan_fp)


In [12]:
# ---------------------------
# Prepare Features and Labels
# ---------------------------
X_list = pd.DataFrame(df1['morgan_fp'].tolist())
scaler = StandardScaler()
X = scaler.fit_transform(X_list)
y = df1['CX Basic pKa']

In [13]:
# optional: split off a final test set (the BO uses CV on the train split)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Output shapes
print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (5280, 1024) (5280,)
Test: (587, 1024) (587,)


In [14]:
# Define search space
xgb_search_space = {
    'n_estimators': (100, 1000),
    'max_depth': (3, 10),
    'learning_rate': (0.01, 0.3, 'log-uniform'),
    'subsample': (0.5, 1.0),
    'colsample_bytree': (0.5, 1.0),
    'gamma':(0, 5),
    'reg_alpha': (0, 5, 'uniform'),
    'reg_lambda': (0, 5, 'uniform')
}

xgb = XGBRegressor(
    objective='reg:squarederror', 
    random_state=42
)
cv = KFold(
    n_splits=5, 
    shuffle=True, 
    random_state=42)

opt_xgb = BayesSearchCV(
    estimator=xgb,
    search_spaces=xgb_search_space,
    n_iter=120,
    cv=5,
    n_points=20,  
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=0
)

In [15]:
 # Fit search and select best model
opt_xgb.fit(X_train, y_train)
best_model = opt_xgb.best_estimator_

In [21]:
# -------------------- CV metrics on training set (5-fold) --------------------
cv_scoring = {'r2': 'r2', 'mse': 'neg_mean_squared_error', 'mae': 'neg_mean_absolute_error'}
cvcv_results = cross_validate(best_model, X_train, y_train, cv=cv, scoring=cv_scoring, n_jobs=-1, return_train_score=False)

cv_mse = -cv_results['test_mse'].mean()
cv_rmse = np.sqrt(cv_mse)
cv_mae = -cv_results['test_mae'].mean()
cv_r2 = cv_results['test_r2'].mean()

In [22]:
xgb_pred = opt_xgb.predict(X_test)
xgb_pred_train = opt_xgb.predict(X_train)

xgb_r2 = r2_score(y_test, xgb_pred)
xgb_r2_train = r2_score(y_train, xgb_pred_train)

xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mae_train = mean_absolute_error(y_train, xgb_pred_train)

xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_mse_train = mean_squared_error(y_train, xgb_pred_train)

In [23]:
print("XGB Best Params:", best_model)
print("XGB MSE:", xgb_mse)
print("XGB MSE Train:", xgb_mse_train)

print("XGB MAE:", xgb_mae)
print("XGB MAE Train:", xgb_mae_train)

print("XGB R2:", xgb_r2)
print("XGB R2 Train:", xgb_r2_train)

XGB Best Params: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.5, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.062098995682300324, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=10, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None,
             num_parallel_tree=None, ...)
XGB MSE: 0.2608728258367951
XGB MSE Train: 0.023849972209445642
XGB MAE: 0.2908883678770959
XGB MAE Train: 0.09553289513967253
XGB R2: 0.8606960804556042
XGB R2 Train: 0.9875346388268749


In [24]:
print(xgb_mae)
print(xgb_mae_train)
print(cv_mae)

print(xgb_mse)
print(xgb_mse_train)
print(cv_mse)


print(xgb_r2)
print(xgb_r2_train)
print(cv_r2)

0.2908883678770959
0.09553289513967253
0.3367620621928663
0.2608728258367951
0.023849972209445642
0.3418657028446516
0.8606960804556042
0.9875346388268749
0.8212430551770952
